# Week 6, Day 4 — Assembling the Trading Floor
### Local Models Edition — by Abhishek

Today the real `backend/` package comes together: `Trader` (in
`backend/traders.py`) uses its Accounts, Push and Market MCP servers plus a
Researcher sub-agent (Fetch, free search, and Memory), all wired up in
`backend/mcp_servers.py`.

**Local/free throughout:** `get_model()` in `backend/traders.py` has an
`ollama:` branch pointing at your local Ollama server — that's the only
change needed to run the *exact same* trader/researcher architecture from
the original course for free.


## 0. Setup — pull a local model first

In [ ]:
import subprocess
print(subprocess.run(["node", "--version"], capture_output=True, text=True).stdout or "Node not found - install from nodejs.org")


In [ ]:
%pip install -q openai-agents mcp duckduckgo-search python-dotenv


## Reset the trading floor

`backend/reset.py` holds the four real trader personas — Warren (value,
after Warren Buffett), George (macro, after George Soros), Ray (risk parity,
after Ray Dalio), and Cathie (disruptive/crypto growth, after Cathie Wood).
Unchanged from the original course — great strategy prompts don't need a
paid model to be worth teaching.


In [ ]:
from backend.reset import reset_traders
reset_traders()

from backend.accounts import Account
for name in ["Warren", "George", "Ray", "Cathie"]:
    print(Account.get(name).report())


## Run one trader

`Trader.run()` (in `backend/traders.py`) builds the researcher tool, builds
the trading agent with its MCP servers, reads the account and strategy, and
runs the whole thing inside a trace — all exactly as in the original course.


In [ ]:
from backend.traders import Trader

warren = Trader("Warren", "Patience", "ollama:llama3.2:3b")
await warren.run()
print("Warren's turn is complete - see the account below")

import json
print(json.loads(await warren.get_account_report()))


## Inspecting the trace log

`backend/tracers.py`'s `LogTracer` was already fully local in the original
course — every span writes straight to `accounts.db` via `database.py`. No
OpenAI account or platform tracing required, on local models or otherwise.


In [ ]:
from backend.database import read_log
for timestamp, kind, message in read_log("Warren", last_n=15):
    print(f"[{kind}] {timestamp}: {message}")


## Recap, and where we are heading

You ran a real trader from the original course's own `backend/traders.py`,
unmodified apart from `get_model()` gaining a local branch — researching
with free tools, trading through the accounts MCP server, and logging every
step locally.

Tomorrow: all four traders together, on a schedule, watched through the
real Gradio dashboard from `demo/ui.py`.

## Exercise
Run George, Ray, and Cathie the same way and compare: does each persona's
strategy visibly shape its research questions and trades? Then try
`USE_MANY_MODELS=true` behavior manually — give two traders different local
models (e.g. `ollama:llama3.2:3b` vs `ollama:qwen2.5:3b`) and compare.
